In [1]:
# Step 1: Setup with Dataset Verification
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.metrics import classification_report, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split
import os

def load_or_simulate_dataset(dataset_name, default_rows=1000):
    """Try loading dataset or simulate reasonable defaults"""
    try:
        if dataset_name == "diabetes":
            return pd.read_csv('/kaggle/input/pima-indians-diabetes-database/diabetes.csv')
        elif dataset_name == "arrhythmia":
            return pd.read_csv('/kaggle/input/mitbih-arrhythmia-database/mitbih_train.csv', header=None)
        elif dataset_name == "stress":
            return pd.read_csv('/kaggle/input/stress-detection-in-employees/Stress.csv')  # Common filename
        else:
            raise FileNotFoundError
    except:
        print(f"⚠️ {dataset_name} dataset not found - simulating data")
        return simulate_dataset(dataset_name, default_rows)

def simulate_dataset(dataset_name, rows):
    """Create realistic synthetic data"""
    if dataset_name == "diabetes":
        data = pd.DataFrame({
            'Pregnancies': np.random.randint(0, 10, rows),
            'Glucose': np.random.normal(120, 30, rows).clip(70, 200),
            'BloodPressure': np.random.normal(70, 15, rows).clip(40, 100),
            'SkinThickness': np.random.normal(20, 5, rows),
            'Insulin': np.random.normal(100, 50, rows),
            'BMI': np.random.normal(30, 5, rows),
            'DiabetesPedigreeFunction': np.random.uniform(0.1, 1.5, rows),
            'Age': np.random.randint(20, 70, rows),
            'Outcome': np.random.binomial(1, 0.3, rows)
        })
        data['hrv'] = np.where(data['Outcome'] == 1, 
                             np.random.normal(50, 5, rows),
                             np.random.normal(70, 5, rows))
    
    elif dataset_name == "arrhythmia":
        data = pd.DataFrame(np.random.randn(rows, 187))  # MIT-BIH has 187 timesteps
        data['label'] = np.random.randint(0, 5, rows)  # 5 classes
    
    elif dataset_name == "stress":
        data = pd.DataFrame({
            'snoring_range': np.random.normal(50, 10, rows),
            'respiration_rate': np.random.normal(20, 3, rows),
            'body_temp': np.random.normal(98, 1, rows),
            'limb_movement': np.random.normal(5, 2, rows),
            'blood_oxygen': np.random.normal(97, 2, rows),
            'eye_movement': np.random.normal(5, 2, rows),
            'sleeping_hours': np.random.normal(6, 1, rows),
            'heart_rate': np.random.normal(70, 10, rows),
            'stress': np.random.binomial(1, 0.25, rows)
        })
        data['gsr'] = np.where(data['stress'] == 1, 
                             np.random.normal(8, 1, rows),
                             np.random.normal(4, 1, rows))
    
    return data

# Load or simulate all datasets
diabetes_data = load_or_simulate_dataset("diabetes")
arrhythmia_data = load_or_simulate_dataset("arrhythmia") 
stress_data = load_or_simulate_dataset("stress")

# Verify datasets
print("✅ Loaded datasets:")
print(f"- Diabetes: {diabetes_data.shape}")
print(f"- Arrhythmia: {arrhythmia_data.shape}")
print(f"- Stress: {stress_data.shape}")

⚠️ arrhythmia dataset not found - simulating data
⚠️ stress dataset not found - simulating data
✅ Loaded datasets:
- Diabetes: (768, 9)
- Arrhythmia: (1000, 188)
- Stress: (1000, 10)


In [2]:
# Step 2: Diabetes Model with HRV Simulation
if 'hrv' not in diabetes_data.columns:
    diabetes_data['hrv'] = np.where(diabetes_data['Outcome'] == 1,
                                  np.random.normal(50, 5, len(diabetes_data)),
                                  np.random.normal(70, 5, len(diabetes_data)))

X_dia = diabetes_data[['Glucose', 'BMI', 'Age', 'hrv']]
y_dia = diabetes_data['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X_dia, y_dia, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_dia = RandomForestClassifier(n_estimators=100)
rf_dia.fit(X_train_scaled, y_train)

print("Diabetes Model Performance:")
print(classification_report(y_test, rf_dia.predict(X_test_scaled)))

Diabetes Model Performance:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        99
           1       1.00      0.96      0.98        55

    accuracy                           0.99       154
   macro avg       0.99      0.98      0.99       154
weighted avg       0.99      0.99      0.99       154



In [3]:
# Step 3: Arrhythmia Detection (Works with MIT-BIH or simulated)
arrhythmia_data.columns = [f'ecg_{i}' for i in range(arrhythmia_data.shape[1]-1)] + ['label']

# Simple version for Kaggle (full LSTM would need more preprocessing)
X_arr = arrhythmia_data.iloc[:, :-1]
y_arr = arrhythmia_data['label']

# Binary classification (normal vs abnormal)
y_arr_binary = (y_arr > 0).astype(int)

rf_arr = RandomForestClassifier()
rf_arr.fit(X_arr, y_arr_binary)

print("Arrhythmia Detection (Binary):")
print(classification_report(y_arr_binary, rf_arr.predict(X_arr)))

Arrhythmia Detection (Binary):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       195
           1       1.00      1.00      1.00       805

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [4]:
# Step 4: Stress Detection with GSR Simulation
if 'gsr' not in stress_data.columns:
    stress_data['gsr'] = np.where(stress_data['stress'] == 1,
                                np.random.normal(8, 1, len(stress_data)),
                                np.random.normal(4, 1, len(stress_data)))

X_stress = stress_data[['gsr', 'heart_rate', 'body_temp']]
y_stress = stress_data['stress']

svm_stress = SVC(probability=True)
svm_stress.fit(X_stress, y_stress)

print("Stress Detection Performance:")
print(classification_report(y_stress, svm_stress.predict(X_stress)))

Stress Detection Performance:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       737
           1       0.99      0.70      0.82       263

    accuracy                           0.92      1000
   macro avg       0.95      0.85      0.89      1000
weighted avg       0.93      0.92      0.92      1000



In [5]:
# Step 5: Blood Pressure Simulation (since dataset was missing)
# Create synthetic BP data using known physiological relationships
bp_data = pd.DataFrame({
    'age': np.random.randint(20, 80, 500),
    'bmi': np.random.normal(25, 5, 500).clip(18, 40),
    'heart_rate': np.random.normal(70, 10, 500).clip(50, 100),
    'ptt': np.random.normal(0.15, 0.03, 500)  # Pulse Transit Time
})
bp_data['sbp'] = 110 + 0.5*bp_data['age'] + 0.3*bp_data['bmi'] - 0.2*bp_data['ptt']*1000
bp_data['dbp'] = 70 + 0.3*bp_data['age'] + 0.2*bp_data['bmi'] - 0.1*bp_data['ptt']*1000

# Add simulated ECG quality
bp_data['ecg_quality'] = np.random.uniform(0.8, 1.0, len(bp_data))

# Train BP model
svr_sbp = SVR()
svr_sbp.fit(bp_data[['ptt', 'ecg_quality']], bp_data['sbp'])

print("BP Model MAE:", mean_absolute_error(
    bp_data['sbp'], 
    svr_sbp.predict(bp_data[['ptt', 'ecg_quality']])
))

BP Model MAE: 7.890384023906254


In [6]:
# Step 6: Export for Smartwatch Integration
import joblib

joblib.dump({
    'diabetes_scaler': scaler,
    'diabetes_model': rf_dia,
    'arrhythmia_model': rf_arr,
    'stress_model': svm_stress,
    'bp_model': svr_sbp
}, 'health_models.pkl')

print("✅ All models exported successfully!")

✅ All models exported successfully!
